In [1]:
import ee
import geemap
import pandas as pd
import numpy as np

geemap.ee_initialize()

In [2]:
greenlandmask = ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ocean_mask').eq(0).selfMask()

samples = greenlandmask.sample(numPixels=10, scale=5000, geometries=True)


In [3]:
map = geemap.Map()
map.addLayer(samples, {}, 'samples')
map.centerObject(samples, 4)
map


Map(center=[70.92335801125665, -44.824468089725976], controls=(WidgetControl(options=['position', 'transparent…

In [4]:
# From production script

def bitwiseExtract(input, fromBit, toBit):
    maskSize = ee.Number(1).add(toBit).subtract(fromBit)
    mask = ee.Number(1).leftShift(maskSize).subtract(1)
    return input.rightShift(fromBit).bitwiseAnd(mask)

def maskQualityDaytime(image):
    qa = image.select('QC_Day')
    # Bits 0-"1": Mandatory QA flags
    # "0": LST produced, good quality, not necessary to examine more detailed QA
    # "1": LST produced, other quality, recommend examination of more detailed QA
    # "2": LST not produced due to cloud effects
    # "3": LST not produced primarily due to reasons other than cloud
    bits01Mask = bitwiseExtract(qa, 0, 1).lte(1); 
    # Bits 2-"3": Data quality flag
    # "0": Good data quality
    # "1": Other quality data
    # "2": TBD
    # "3": TBD
    bits23Mask = bitwiseExtract(qa, 2, 3).eq(0)
    # Bits 4-"5": Emissivity error flag
    # "0": Average emissivity error <= 0.01
    # "1": 0.01 < Average emissivity error <= 0.02
    # "2": 0.02 < Average emissivity error <= 0.04
    # "3": Average emissivity error > 0.04
    bits45Mask = bitwiseExtract(qa, 4, 5).eq(0)
    # Bit 6-"7": LST error flag
    # "0": Average LST error <= 1K
    # "1": Average LST error <= 2K
    # "2": Average LST error <= 3K
    # "3": Average LST error > 3K
    bit6Mask = bitwiseExtract(qa, 6, 7).lte(1)

    mask = bits01Mask.And(bits23Mask).And(bits45Mask).And(bit6Mask)

    return image.updateMask(mask)

def maskQualityNighttime(image):
    qa = image.select('QC_Night')

    # Bits 0-"1": Mandatory QA flags
    # "0": LST produced, good quality, not necessary to examine more detailed QA
    # "1": LST produced, other quality, recommend examination of more detailed QA
    # "2": LST not produced due to cloud effects
    # "3": LST not produced primarily due to reasons other than cloud
    bits01Mask = bitwiseExtract(qa, 0, 1).lte(1); 
    # Bits 2-"3": Data quality flag
    # "0": Good data quality
    # "1": Other quality data
    # "2": TBD
    # "3": TBD
    bits23Mask = bitwiseExtract(qa, 2, 3).eq(0)
    # Bits 4-"5": Emissivity error flag
    # "0": Average emissivity error <= 0.01
    # "1": 0.01 < Average emissivity error <= 0.02
    # "2": 0.02 < Average emissivity error <= 0.04
    # "3": Average emissivity error > 0.04
    bits45Mask = bitwiseExtract(qa, 4, 5).eq(0)
    # Bit 6-"7": LST error flag
    # "0": Average LST error <= 1K
    # "1": Average LST error <= 2K
    # "2": Average LST error <= 3K
    # "3": Average LST error > 3K
    bit6Mask = bitwiseExtract(qa, 6, 7).lte(1)

    mask = bits01Mask.And(bits23Mask).And(bits45Mask).And(bit6Mask)

    return image.updateMask(mask)


# MODIS
def lst_mod_day(image):
    'Terra Day band selection and conversion'
    lst_day = image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('MOD_LST_Day')
    return lst_day

def lst_mod_night(image):
    'Terra Night band selection and conversion'
    lst_night = image.select('LST_Night_1km').multiply(0.02).subtract(273.15).rename('MOD_LST_Night')
    return lst_night

def lst_myd_day(image):
    'Aqua Day band selection and conversion'
    lst_day = image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('MYD_LST_Day')
    return lst_day

def lst_myd_night(image):
    'Aqua Night band selection and conversion'
    lst_night = image.select('LST_Night_1km').multiply(0.02).subtract(273.15).rename('MYD_LST_Night')
    return lst_night

In [6]:
poi = samples
date_start = '2020-01-01'
date_end = '2024-12-31'

MOD11A1Daytime = (
    ee.ImageCollection('MODIS/061/MOD11A1')
    .select(['LST_Day_1km', 'QC_Day'])
    .filterDate(date_start, date_end)
    .filterBounds(poi)
    .map(maskQualityDaytime)
    .map(lst_mod_day)
)

MOD11A1Nighttime = (
    ee.ImageCollection('MODIS/061/MOD11A1')
    .select(['LST_Night_1km', 'QC_Night'])
    .filterDate(date_start, date_end)
    .filterBounds(poi)
    .map(maskQualityNighttime)
    .map(lst_mod_night)
)

MOD = MOD11A1Daytime.merge(MOD11A1Nighttime)

MYD11A1Daytime = (
    ee.ImageCollection('MODIS/061/MYD11A1')
    .select(['LST_Day_1km', 'QC_Day'])
    .filterDate(date_start, date_end)
    .filterBounds(poi)
    .map(maskQualityDaytime)
    .map(lst_myd_day)
)

MYD11A1Nighttime = (
    ee.ImageCollection('MODIS/061/MYD11A1')
    .select(['LST_Night_1km', 'QC_Night'])
    .filterDate(date_start, date_end)
    .filterBounds(poi)
    .map(maskQualityNighttime)
    .map(lst_myd_night)
)

MYD = MYD11A1Daytime.merge(MYD11A1Nighttime)

In [7]:
## Map over the ImageCollection
def per_image(img):
    return img.reduceRegions(
        collection=poi,
        reducer= ee.Reducer.mean(),
        scale= 1000,
        crs='EPSG:3413'
    )

# Add the date and id of the poi to the results
def add_properties(img):
    date = img.date().format('YYYY-MM-dd')
    return img.set('date', date)

results_mod = MOD.map(per_image).flatten().map(add_properties)
results_myd = MYD.map(per_image).flatten().map(add_properties)

# Convert to pandas DataFrame
MOD_df = geemap.ee_to_df(results_mod)
MYD_df = geemap.ee_to_df(results_myd)

print(MOD_df.head())
print(MYD_df.head())

AttributeError: 'Feature' object has no attribute 'date'